# Stage 3 — GSE270828 full guide reference retrieval

**One-action Colab utility.** Downloads the full original GEO feature reference (not a preview), validates its schema and records its checksum, and exports a small ZIP to share in ChatGPT.

**Important:** The file contains guide/target *metadata*, NOT experimentally measured gene-level treatment effects. The target_gene_name column cannot be treated as validated perturbation evidence.

In [ ]:
import requests, io, time, json, hashlib, zipfile, pandas as pd
from pathlib import Path
from IPython.display import display

urls = [
 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/GSE270828_feature_README.csv',
 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/GSE270828_feature_README.csv?download=1',
]
output = Path('/content/GSE270828_feature_README.csv')
errors = []
for url in urls:
    try:
        res=requests.get(url,timeout=50,headers={'User-Agent':'Har-Research-Reproducibility/1.0'})
        res.raise_for_status()
        body=res.content
        if len(body)<1000 or not body.startswith(b'id,'):
            raise ValueError('Unexpected file header or very short payload: '+repr(body[:100]))
        output.write_bytes(body)
        print('Original feature reference downloaded:',len(body),'bytes')
        break
    except Exception as e:
        errors.append(str(e))
else:
    raise RuntimeError('Download failed; open direct GEO URL in browser. Errors: '+str(errors))

data=pd.read_csv(output)
print('Columns:',data.columns.tolist())
display(data.head(8))
required={'id','name','sequence','target_gene_name'}
if not required.issubset(set(data.columns)):
    raise ValueError('Unexpected columns; do not assume target mapping. Found '+str(list(data.columns)))
summary={
 'source_accession':'GSE270828',
 'source_url':urls[0],
 'size_bytes':output.stat().st_size,
 'sha256':hashlib.sha256(output.read_bytes()).hexdigest(),
 'total_feature_rows':len(data),
 'unique_feature_ids':int(data.id.nunique()),
 'unique_har_labels':int(data.name.nunique()),
 'unique_design_target_gene_names':int(data.target_gene_name.nunique()),
 'important_interpretation':'guide design metadata; not functional gene-target validation',
}
print(json.dumps(summary,indent=2))
json_path=Path('/content/GSE270828_guide_reference_summary.json')
json_path.write_text(json.dumps(summary,indent=2))
zip_path=Path('/content/GSE270828_Full_Guide_Reference_FOR_REVIEW.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(output,output.name)
    archive.write(json_path,json_path.name)
print('READY:',zip_path)
try:
 from google.colab import files
 files.download(str(zip_path))
except ImportError:
 print('Not in Colab; download file manually:',str(zip_path))

### Next step
Upload the downloaded `GSE270828_Full_Guide_Reference_FOR_REVIEW.zip` in the same ChatGPT conversation. We will match guide HAR names to Pal Capture-HiC bait labels, report candidate coverage, and identify the additional effect data needed for a valid benchmark.